## **00. Creación del dataset**

## **Proyecto final de 4Geeks: Sistema de alertas de seguridad urbana**
- **Equipo:** Alessandra | Natalia | Andrés  
- **Objetivo de este notebook:** explicar de dónde vienen los datos, cuáles datasets combinamos, cómo los unificamos y qué significa cada columna del dataset final.

### **1. Planteamiento del problema**
**Contexto**

Los espacios públicos en las ciudades, como las calles, las plazas, el transporte público y los aparcamientos, son lugares donde hay muchos ruidos diferentes. Los ruidos cotidianos conviven con situaciones de peligro real. Ser capaz de distinguir automáticamente entre un perro ladrando, niños jugando y un disparo tiene aplicaciones directas en:
- **Seguridad municipal:** detectar disparos o accidentes de forma automática.
- **Accesibilidad:** alertas visuales o de vibración para personas con discapacidades auditivas.
- **Sistemas de seguridad inteligentes:** integrar audio con cámaras de seguridad municipales.
- **Redes de vigilancia municipal y aplicaciones de seguridad ciudadana:** notificar a los usuarios en zonas peligrosas en tiempo real.
- **Seguros y análisis forense:** registrar y clasificar incidentes de forma automática.

Nuestro proyecto propone entrenar un modelo de clasificación de audio que identifique sonidos urbanos peligrosos o relevantes, con el objetivo de enviar alertas automáticas. 

Dado un fragmento de audio de corta duración grabado en un entorno urbano, nuestro modelo podrá **clasificar qué tipo de sonido es** y determinar **si representa una situación de emergencia o de alerta**.

Este es un problema de **clasificación multiclase supervisada** basado en señales de audio. La variable objetivo principal es la clase del sonido (`human_label`).

En principio, este será el plan. De ser necesario, se entrenará un modelo híbrido que pueda diferenciar entre sonidos alertables y no alertables y, luego, un modelo más específico para identificar qué tipo de sonido es.

### **2. Objetivos**

**Objetivo general**

Desarrollar un sistema inteligente basado en el aprendizaje automático capaz de detectar y clasificar sonidos ambientales para generar alertas accesibles orientadas al apoyo de personas con discapacidad auditiva y a la identificación de situaciones relevantes del entorno urbano. 

**Objetivos específicos**
- Entrenar modelos de Machine Learning y Deep Learning para la clasificación automática de sonidos con una precisión y un recall (f1-score) mayores al 60 %.
- Implementar un módulo de detección automática capaz de analizar audios con una duración de entre 1 y 5 segundos.
- Desarrollar un sistema de generación de alertas visuales y digitales basado en los sonidos detectados.
- Validar el funcionamiento del sistema mediante pruebas experimentales con diferentes tipos de sonidos ambientales.
- Preparar el sistema para futuras ampliaciones mediante la integración de nuevos datasets y sensores acústicos.

### **3. Fuentes de datos**

Combinamos varios datasets públicos complementarios para obtener **≥ 100.000 filas** y **20 columnas**. Cada dataset aporta clases y contextos sonoros distintos que enriquecen el modelo final.

### **3.1 UrbanSound8K (Kaggle)**
**Fuente:** https://www.kaggle.com/datasets/chrisfilo/urbansound8k

Este dataset contiene **8.732 fragmentos** de audio de hasta 4 segundos, etiquetados y extraídos de grabaciones reales subidas a la plataforma Freesound. Cada fragmento pertenece a una de 10 clases de sonidos urbanos y tiene un archivo de metadatos llamado UrbanSound8K.csv. Es el dataset más citado en la literatura de clasificación de audio urbano.

| Clase | Relevancia para el sistema de alertas |
|---|---|
| `gun_shot` | crítica |
| `siren` | crítica |
| `car_horn` | alta |
| `dog_bark` | media |
| `drilling` | media |
| `jackhammer` | media |
| `engine_idling` | baja |
| `children_playing` | baja |
| `street_music` | baja |
| `air_conditioner` | baja |

Los autores del dataset advierten que no debemos reordenar los datos. El dataset viene ya dividido en 10 folds y los fragmentos del mismo archivo original siempre están en el mismo fold, evitando data leakage. Seguimos la **validación cruzada de 10 folds** con las divisiones predefinidas.

### **3.2 FSD50K (Zenodo)**
**Fuente:** https://zenodo.org/records/4060432

Este dataset contiene **51.197 fragmentos** de audio de Freesound distribuidos de forma desigual en 200 clases extraídas de la ontología AudioSet. FSD50K fue creado en el Grupo de Tecnología Musical (MTG) de la Universitat Pompeu Fabra.

### **3.3 ESC-50 (GitHub)**
**Fuente:** https://github.com/karolpiczak/ESC-50  

El dataset ESC-50 es una colección etiquetada de **2.000 grabaciones de audio ambiental**, ideal para evaluar métodos de clasificación de sonidos ambientales. Consta de grabaciones de 5 segundos de duración, organizadas en 50 clases (con 40 ejemplos por clase) y agrupadas en 5 categorías: animales, ruidos naturales, interior, exterior y actividades humanas. Complementa los datasets anteriores con sonidos del entorno cotidiano (pasos, lluvia, reloj, etc.), que añaden contexto al modelo.

### **3.4 AudioSet (HuggingFace)**
**Fuente:** https://huggingface.co/datasets/agkphysics/AudioSet  

Se trata de la versión completa de AudioSet, accesible a través de HuggingFace. Lo usamos para complementar las clases subrepresentadas en el dataset de Zenodo, especialmente las categorías de emergencia (`siren`, `gunshot`, `alarm`).

### **3.5 Gunshot-audio-dataset (Kaggle)**
**Fuente:** https://www.kaggle.com/datasets/huseyngorbani1/gunshot-audio-dataset?select=gunshot-audio-dataset

Este es un dataset de clips de audio de disparos recopilados de vídeos de YouTube, con **9 clases** correspondientes a modelos de armas específicos: IMI Desert Eagle, M4, MP5, MG-42, M16, M249, AK-12, Zastava M92 y AK-47. Lo usamos para reforzar la clase gunshot, que es crítica para el sistema de alertas. En este dataset hay un total de **851 archivos**. Los archivos mezclan frecuencias de muestreo de **44100 Hz y 48000 Hz**, y todos son stereo.

### **3.6 VOICe Dataset (Zenodo)**
**Fuente:** https://zenodo.org/records/3514950

Contiene 1449 mezclas de tres eventos sonoros: llanto de bebé, rotura de cristal y disparos. A su vez, están mezclados con ruido de fondo de tres escenas acústicas: vehículos, ruido exterior y ruido interior. Es especialmente útil para el proyecto porque simula condiciones reales de detección, ya que los sonidos de emergencia no aparecen en silencio sino mezclados con el ruido del entorno.

### **3.7 Sound Event Detection for Driver Safety (Kaggle)**
**Fuente:** https://www.kaggle.com/datasets/ccastorena/sound-event-detection-for-driver-safety

Se trata de un dataset sintético de **19.000 clips de audio** de **10 segundos** diseñado para la detección de eventos sonoros en escenarios de conducción. Incluye **9 clases** seleccionadas como distractores auditivos relevantes para la seguridad vial: Ring Tone, Vibrating, Notification, Speech, Baby Cry, Physiological, Pets, Horns y Sirens. Fue publicado junto a un artículo científico en el que se propone un framework basado en CRNN y YOLO.

### **3.8 Emergencysound (Kaggle)**
**Fuente:** https://www.kaggle.com/datasets/buraktaci/emergencysound

Dataset compuesto por **4.841 archivos WAV** con duración media aproximada de **1 segundo**, organizado en 4 clases orientadas a la detección de situaciones de emergencia: Crying, High intensity, Low intensity y Violence. Es útil para sistemas de alerta temprana basados en el nivel y la intensidad del sonido.

### **3.9 Enhanced audio of accident and crime detection (Kaggle)**
**Fuente:** https://www.kaggle.com/datasets/afisarsy/enhanced-audio-of-accident-and-crime-detection

Dataset de **9.089 archivos WAV de duración variable**, organizado en **13 clases** orientadas a la detección de accidentes y crímenes en entornos urbanos. Es la versión mejorada (preprocesada y aumentada) del dataset en bruto del mismo autor (`raw-audio-of-accident-and-crime-detection`).

### **3.10 Emergency Vehicle Siren Sounds (Kaggle)**
**Fuente:** https://www.kaggle.com/datasets/vishnu0399/emergency-vehicle-siren-sounds

Dataset compuesto por clips WAV de **3 segundos** con sirenas de vehículos de emergencia (ambulancia y camión de bomberos) y una tercera categoría de sonido de tráfico ordinario. Cada categoría contiene 200 archivos de audio junto con sus espectrogramas correspondientes en formato PNG. Contiene un total de **600 clips de audio**.

### **3.11 Vídeos varios (YouTube)**
**Fuente:** Elaboración propia (Andrés)

Este dataset contiene **6.974 segmentos de audio de 5 segundos** extraídos de vídeos de YouTube. El procesamiento consistió en descargar los primeros 30 minutos de cada vídeo con yt-dlp, segmentar el audio en clips de 5 segundos con librosa y filtrar por energía mínima para eliminar silencios. De cada segmento se extrajeron 6 features acústicas (peak, rms, centroid, zcr, kurtosis, attack_ratio). Se está utilizando para reforzar las 5 clases del modelo (`fire`, `gunshot`, `explosion`, `car_crash` y `crying`) con audio procedente de contextos reales y variados.

### **3.12 Edge-collected-gunshot dataset (CSV)**
**Fuente:** Elaboración propia (Andrés)

Este dataset contiene **2.148 grabaciones** de disparos reales distribuidas en 4 clases (`9mm`, `38cal`, `AR15`, `12gauge`), capturadas durante 2 sesiones de grabación en un campo de tiro. Cada registro incluye metadatos del dispositivo de captura, coordenadas GPS, una marca temporal y la localización exacta de cada disparo dentro del audio. Se está utilizando este dataset para reforzar la clase `gunshot` en el dataset final del modelo.

### **Resumen de fuentes**

| Dataset | Clips | Duración media | Clases | Responsable |
|---|---|---|---|---|
| UrbanSound8K (Kaggle) | 8.732 | 4s | 10 | Alessandra |
| FSD50K (Zenodo) | 51.197 | variable (de 0.3s a 30s) | filtradas | Andrés |
| ESC-50 (GitHub) | 2.000 | 5s | 50 | Natalia |
| AudioSet (HuggingFace) | complementario | 10s | filtradas | Natalia |
| Gunshot-audio-dataset (Kaggle) | 851 | 3.04s | 9 armas | — |
| VOICe Dataset (Zenodo) | 1.449 | variable | 3 | — |
| Sound Event Detection for Driver Safety (Kaggle) | 19.000 | 10s | 9 | — |
| Emergencysound (Kaggle) | 4.841 | 1s | 4 | — |
| Enhanced audio of accident and crime detection (Kaggle) | 9.089 | variable | 13 | — |
| Emergency Vehicle Siren Sounds (Kaggle) | 600 | 3s | 3 (ambulancia, bomberos, tráfico) | — |
| Vídeos varios (YouTube) | 6.974 | 5s | 5 | Andrés |
| Edge-collected-gunshot dataset (CSV) | 2.148 | variable | 1 (gunshot) con 4 subtipos de arma | Andrés |
| **Total** | **≥ 100.000** |  | | |

### **4. Estructura del proyecto**

```
proyecto4geeks/
│
├── data/
│   ├── raw/ --> Audios y CSVs originales sin modificar
│   │   ├── urbansound8k/
│   │   ├── zenodo/
│   │   ├── esc50/
│   │   └── audioset/
│   ├── interim/ --> Audios filtrados + CSVs homogeneizados por dataset
│   │   ├── urbansound8k/
│   │   ├── zenodo/
│   │   ├── esc50/
│   │   └── audioset/
│   └── processed/ --> Dataset final listo para tanto para el EDA como para el entrenamiento del modelo
│
├── src/data/build/
│   ├── dataset.py --> Clase Dataset y utilidades generales
│   ├── metadata.py (MetadataEX) --> Extracción de metadatos de audio
│   ├── urbansound8k/ --> Scripts específicos de UrbanSound8K
│   ├── zenodo_ds/ --> Scripts específicos de AudioSet/Zenodo
│   ├── esc50/ --> Scripts específicos de ESC-50
│   └── audioset/ --> Scripts específicos de AudioSet HuggingFace
│
├── notebooks/
│   ├── 00_building_ds.ipynb --> Este notebook
│   ├── 01_eda.ipynb
│   ├── 02_preprocessing.ipynb
│   ├── 03_modeling.ipynb
│   └── 04_evaluation.ipynb
│
├── deployment/ --> App web (Flask / Streamlit)
├── experiments/ --> Resultados de nuestros experimentos y métricas
└── src/ --> Código fuente del modelo
```

### **5. Proceso de construcción del dataset**

El proceso de construcción del dataset final sigue estos pasos para cada fuente:

```
[Descarga]  →  data/raw/<dataset>/
            ↓
[Filtrado de clases relevantes]  
            ↓
[Extracción de metadatos de audio]  
            ↓
[Generación de CSV homogeneizado]  
            ↓
[Unión de todos los CSVs]  
```

### **6. Descripción de las columnas del dataset final**
El dataset final contiene **20 columnas**, cumpliendo el requisito del proyecto. Incluye variables de identificación, de etiquetado, de seguridad y metadatos técnicos del audio.

### **6.1 Variables de identificación**

| Columna | Tipo | Ejemplo | Descripción |
|---|---|---|---|
| `audio` | str | `64760.wav` | Nombre del archivo de audio tal como viene en el dataset original. |
| `audio_format` | str | `*.WAV`, `*.m4a` | Formato del archivo. |
| `path` | str | `ESC50/1-115545-A-48.wav` | Carpeta donde está almacenado el audio. |
| `dataset_source` | str | `zenodo` | Dataset de origen del fragmento: `urbansound8k`, `zenodo`, `esc50` o `audioset`. Permite filtrar por fuente. |
| `file_exists` | bool | `True`, `False` | Columna de control para verificar que los archivos están en disco durante la creación del dataset. |

### **6.2 Variables de etiquetado**

| Columna | Tipo | Ejemplo | Descripción |
|---|---|---|---|
| `label` | str | `/m/02sgy` | Identificador de clase en el vocabulario original del dataset fuente.|
| `human_label` | str | `guitar` | Clase principal homogeneizada en lenguaje natural. **Es la columna más importante para el modelo y la que se muestra en la aplicación web.** |
| `labels` | list[str] | `["/m/02sgy", "/m/0342h", "/m/0fx80y", "/m/04szw", "/m/04rlf"]` | Lista completa de todos los identificadores de clase del clip en el vocabulario original. Un clip puede tener múltiples sonidos simultáneos. |
| `human_labels` | list[str] | `["guitar", "strings", "Musical_instrument", "Music"]` | Lista completa de todas las clases homogeneizadas del clip. |

### **6.3 Variables de seguridad**

| Columna | Tipo | Valores | Descripción |
|---|---|---|---|
| `emergency` | bool | `True`, `False` | Permite formular el problema como una **clasificación binaria** (¿es o no es una emergencia?). |
| `alertable` | bool | `True`, `False` | ¿Es una situación que requiera una alerta o notificación sin ser una emergencia? |
| `env` | str | `exterior`, `interior`, `indefinida` | Categoría de entorno sonoro inferida de la clase. Ejemplos: `traffic` --> `exterior`, `doors` --> `interior`, `guitar` --> `indefinida`. Es útil para contextualizar las alertas. |

### **6.4 Variables de partición**

| Columna | Tipo | Valores | Descripción |
|---|---|---|---|
| `split` | str | `train`, `test` | Conjunto de entrenamiento o prueba al que pertenece el fragmento. En UrbanSound8K, se respetan los 10 folds originales. En el resto, se aplica una división 80/20 por clase para evitar el desbalanceo. |

### **6.5 Metadatos técnicos del audio**

| Columna | Tipo | Ejemplo | Descripción |
|---|---|---|---|
| `duration` | float | `4.0` | Duración del fragmento en segundos. Varía entre datasets: UrbanSound8K 4s, AudioSet 10s.|
| `sample_rate` | int | `48000` | Frecuencia de muestreo en Hz. |
| `channels` | str | `Mono` | Número de canales: mono o estéreo.|
| `bit_depth` | int | `160` | Profundidad de bits de la señal (bits por muestra). Indica la resolución de amplitud. |
| `bit_velocity` | float | `1411 kbps` | Tasa de bits en kbps (kbits/segundo). Se calcula como `sample_rate × bit_depth × channels / 1000`. Indica la calidad y el peso del audio. |
| `size` | float | `43071` | Tamaño del archivo en kilobytes. Es útil para detectar clips muy pequeños o muy grandes. |
| `date_modification` | datetime | `04/02/2020 18:40` | Fecha de última modificación del fichero. |

### **Variables que NO usamos como features del modelo**
- **`label`, `labels`**: son los IDs en su vocabulario original.
- **`date_modification`**: metadato de gestión; no tiene valor predictivo.
- **`bit_velocity`**: se puede derivar de `sample_rate`, `bit_depth` y `channels`; se utiliza en el preprocesamiento.

### **7. Constucción del dataset: código**

src\data\buildsrc\data\build

In [ ]:
import pandas as pd
import os
from pathlib import Path
from src.data.build.dataset import Dataset
from src.data.build.metadata import MetadataEX
from src.utils.config import *
from src.data.build.audioset import AudioSet
from src.data.build.dataset import extraer_eventos

### **7.1 UrbanSound8K (Kaggle)**

In [ ]:
script_dir =  BUILD_DIR / "UrbanSound8K"
file_list = {"UrbanSound8k.csv":["UrbanSound8K.csv",0.8]}
df_res={}
for key,val in file_list.items():
    df_original = pd.read_csv(os.path.join(script_dir, val[0]))
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=df_original,
        col_labels='class',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='slice_file_name',
        ruta_carpeta="UrbanSound8k/",
        sinonimos=None,
        nombre_salida=nombre_salida,
        dataset_name="UrbanSound8k",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "UrbanSound8k.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="UrbanSound8k",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.2 FSD50K (Zenodo)**

In [ ]:
script_dir =  BUILD_DIR / "zenodo_ds"
file_list = {"zenodo_train.csv":["dev.csv","train"],"zenodo_test.csv":["eval.csv","test"]}
df_res={}
for key,val in file_list.items():
    df_original = pd.read_csv(os.path.join(script_dir, val[0]))
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=df_original,
        col_labels='labels',
        col_mids='mids',
        col_fname='fname',
        ruta_carpeta="zenodo/",
        sinonimos=None,
        nombre_salida=nombre_salida,
        dataset_name="zenodo",
        split=val[1]
    )
    print(df_res[key].head())
archivos=[RAW_DIR / "zenodo_train.csv", RAW_DIR / "zenodo_test.csv"]
Dataset.concatenar_y_ordenar_csvs(archivos ,"audio" , RAW_DIR / "zenodo.csv",delete_old=True)

nombre_salida = os.path.join(RAW_DIR, "zenodo.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="zenodo",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.3 ESC-50 (GitHub)**

In [ ]:
script_dir =  BUILD_DIR / "ESC50"
file_list = {"ESC50.csv":["ESC50.csv",0.8]}
df_res={}
for key,val in file_list.items():
    df_original = pd.read_csv(os.path.join(script_dir, val[0]))
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=df_original,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="ESC50/",
        sinonimos=None,
        nombre_salida=nombre_salida,
        dataset_name="ESC50",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "ESC50.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="ESC50",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.4 AudioSet (HuggingFace)**

In [ ]:
script_dir =  BUILD_DIR / "audioset"
file_list = {"audioset.csv":["audioset.csv",0.8]}

print("Cargando sinonimos...")
sinonimos = pd.read_csv(SINONIMOS_V3_PATH)

print("Cargando canonical clases...")
canonical_df = pd.read_csv(CANONICAL_CLASSES_PATH)

sinonimos["canonical"] = sinonimos["canonical"].str.lower()
sinonimos["synonym"] = sinonimos["synonym"].str.lower()
canonical_df["canonical"] = canonical_df["canonical"].str.lower()

synonym_to_canonical = dict(
    zip(sinonimos["synonym"], sinonimos["canonical"])
)

canonical_info = canonical_df.set_index("canonical").to_dict("index")
AudioSet.build_dataset()
nombre_salida = os.path.join(RAW_DIR, "audioset.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="audioset",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.5 Gunshot-audio-dataset (Kaggle)**

In [ ]:
script_dir =  BUILD_DIR / "Guns_DS"
file_list = {"Guns_DS.csv":["Guns_DS.csv",0.8]}
df_res={}
for key,val in file_list.items():
    
    # Clases que queremos conservar (ahora podemos usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="Guns_DS/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        dataset_name="Guns_DS",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "Guns_DS.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="Guns_DS",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.6 VOICe Dataset (Zenodo)**

In [ ]:
script_dir =  BUILD_DIR / "VOICe"
file_list = {"VOICe.csv":["VOICe.csv",0.8]}
df_res={}
for key,val in file_list.items():
    
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='class',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='slice_file_name',
        ruta_carpeta="VOICe/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        dataset_name="VOICe",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "VOICe.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="VOICe",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.7 Sound Event Detection for Driver Safety (Kaggle)**

In [ ]:
dataset_name ="driver_safety"
script_dir =  BUILD_DIR / dataset_name
file_list = {f"{dataset_name}_train.csv":["train.tsv","train"],f"{dataset_name}_test.csv":["test.tsv","test"]}
columns_to_filter=[
    "Drift",
    "Hail",
    "Hit",
    "Firefighters",
    "Dog",
    "CarHorn",
    "TruckHorn",
    "Cough",
    "Ambulance",
    "Cry",
    "Police",
    "BoatHorn",
    "Yawn",
    "Cat",
    "Notifications",
    "RingTone",
    "TrainHorn",
    "WarningBeeps",
    "Thunder",
    "Motorcyle",
    "Airplane",
    "Train",
    "Truck",
    "Ignition",
    "Door",
    "Scream",
    "Window",
    "Rain",
    "Crash",
    "ManipulatingObjects",
    "Car",
    "Helicopter",
]
df_res={}

for key,val in file_list.items():
    df_original = pd.read_csv(os.path.join(script_dir, val[0]),delimiter="\t")
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    df_filtrado = df_original[df_original["event_label"].isin(columns_to_filter)]
    nombre_salida = os.path.join(RAW_DIR, key)
    carpeta_audio = BUILD_DIR / dataset_name / "audios" / val[1]
    audio_salida = RAW_DIR / dataset_name / val[1]
    os.makedirs(carpeta_audio, exist_ok=True)
    val[0] = val[1]+".csv"
    print("-------------Iniciando corte de audios------------")
    extraer_eventos(
        carpeta_audio=carpeta_audio,
        df=df_filtrado,
        col_file="filename",
        col_onset="onset",
        col_offset="offset",
        col_label="event_label",
        carpeta_salida=audio_salida,
        csv_salida_path=script_dir / val[0]
    )
    df_filtrado = pd.read_csv(os.path.join(script_dir, val[0]))
    print("-------------Generando CSV audios------------")
    df_res[key] = Dataset.generar_csv_audio(
        df=df_filtrado,
        col_labels='event_label',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='new_filename',
        ruta_carpeta=f"{dataset_name}/",
        sinonimos=None,
        nombre_salida=nombre_salida,
        dataset_name=dataset_name,
        split=val[1]
    )
    print(df_res[key].head())
archivos=[RAW_DIR / f"{dataset_name}_train.csv", RAW_DIR / f"{dataset_name}_test.csv"]
Dataset.concatenar_y_ordenar_csvs(archivos ,"audio" , RAW_DIR / f"{dataset_name}.csv",delete_old=True)

nombre_salida = os.path.join(RAW_DIR, f"{dataset_name}.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name=f"{dataset_name}",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.8 Emergencysound (Kaggle)**

In [ ]:
script_dir =  BUILD_DIR / "emergencysound"
file_list = {"emergencysound.csv":["emergencysound.csv",0.8]}
df_res={}
for key,val in file_list.items():
    
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="emergencysound/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        dataset_name="emergencysound",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "emergencysound.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="emergencysound",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.9 Enhanced audio of accident and crime detection (Kaggle)**

In [ ]:
script_dir =  BUILD_DIR / "Enhanced_audio_of_accident"
file_list = {"Enhanced_audio_of_accident.csv":["Enhanced_audio_of_accident.csv",0.8]}
df_res={}
for key,val in file_list.items():
    
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="Enhanced_audio_of_accident/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        ruta_csv_sinonimos=BUILD_DIR / "sinonimosV4.csv",
        ruta_csv_alert_env=BUILD_DIR / "canonical_clasesV2.csv",
        dataset_name="Enhanced_audio_of_accident",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "Enhanced_audio_of_accident.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="Enhanced_audio_of_accident",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.10 Emergency Vehicle Siren Sounds (Kaggle)**

In [ ]:
script_dir = BUILD_DIR / "emergency-vehicle-siren-sounds"
file_list = {"emergency-vehicle-siren-sounds.csv": ["emergency-vehicle-siren-sounds.csv", 0.8]}
df_res = {}
for key, val in file_list.items():

    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)

    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="emergency-vehicle-siren-sounds/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        dataset_name="emergency-vehicle-siren-sounds",
        split=val[1]
    )
    print(df_res[key].head())

nombre_salida = os.path.join(RAW_DIR, "emergency-vehicle-siren-sounds.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="emergency-vehicle-siren-sounds",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.11 Vídeos varios (YouTube)**

In [ ]:
script_dir =  BUILD_DIR / "youtube"
file_list = {"youtube.csv":["youtube.csv",0.8]}
df_res={}
for key,val in file_list.items():
    
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="youtube/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        dataset_name="youtube",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "youtube.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="youtube",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

### **7.12 Edge-collected-gunshot**

In [ ]:
script_dir =  BUILD_DIR / "edge-collected-gunshot-audio"
file_list = {"edge-collected-gunshot-audio.csv":["edge-collected-gunshot-audio.csv",0.8]}
df_res={}
for key,val in file_list.items():
    
    # Clases que quieres conservar (ahora puedes usar los nombres oficiales)
    
    nombre_salida = os.path.join(RAW_DIR, key)
    df_res[key] = Dataset.generar_csv_audio(
        df=None,
        col_labels='category',
        col_mids=RAW_DIR / "zenodo.csv",
        col_fname='filename',
        ruta_carpeta="edge-collected-gunshot-audio/",
        sinonimos=None,
        nocsv=True,
        nombre_salida=nombre_salida,
        dataset_name="edge-collected-gunshot-audio",
        split=val[1]
    )
    print(df_res[key].head())
nombre_salida = os.path.join(RAW_DIR, "edge-collected-gunshot-audio.csv")
metadata = MetadataEX(
    csv_path=nombre_salida,
    dataset_name="edge-collected-gunshot-audio",
    folder=RAW_DIR
)
metadata.generate_metadata_audio()

- **Nota:** Los audios se descargan por separado. Se espera que ocurra algún error en el Codespaces sin los ficheros.

### **8. Resumen del dataset final**
- **Total de instancias:** ≥ 100.000
- **Variables predictoras:** 20 columnas
- **Variables categóricas:** `human_label`, `human_labels`, `label`, `labels`, `path`, `dataset_source`, `env`, `split`, `audio`, `channels`
- **Variable objetivo principal:** `human_label` (clasificación multiclase)
- **Fuentes combinadas:**
    1. UrbanSound8K (Kaggle)
    2. FSD50K (Zenodo)
    3. ESC-50 (GitHub)
    4. AudioSet (HuggingFace)
    5. Gunshot-audio-dataset (Kaggle)
    6. VOICe Dataset (Zenodo)
    7. Sound Event Detection for Driver Safety (Kaggle)
    8. emergencysound (Kaggle)
    9. Enhanced audio of accident and crime detection (Kaggle)
    10. Emergency Vehicle Siren Sounds (Kaggle)
    11. Vídeos varios (YouTube)
    12. Edge-collected-gunshot dataset (CSV)